In [ ]:
# ── Harness setup (run once) ──────────────────────────────────────────────────
import sys, os, json, pathlib, time

sys.path.insert(0, '/opt/ara/lib')
from tina.client import llm_client, model_fast
from ara_metrics import run_queries_with_template, ndcg_at_k, p95, retriever_dense

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

from elasticsearch import Elasticsearch
es = Elasticsearch(os.environ['ES_URL'], api_key=os.environ['ES_API_KEY'], request_timeout=120)
EMBED_ID     = os.environ.get('ARA_EMBED_ID', '.jina-embeddings-v5-text-small')
CANDIDATE_ID = os.environ.get('ARA_EMBED_CANDIDATE_ID', '.jina-embeddings-v5-text-nano')
TRACES = pathlib.Path('/home/elastic/.traces')
TRACES.mkdir(parents=True, exist_ok=True)

POOL = json.loads(pathlib.Path('/home/elastic/dev-sets/benchmark-pool.json').read_text())
print(f'Harness ready. Pool has {len(POOL)} queries. Current={EMBED_ID} Candidate={CANDIDATE_ID}')

In [ ]:
# ── YOUR WORK ── Assemble your benchmark set ──────────────────────────────────
# Select at least 12 queries from POOL covering all 3 doc types:
#   'policy', 'sar', 'wire-fraud'

# Example: select all — replace with your own selection
MY_BENCHMARK = [q for q in POOL if q['doc_type'] in ('policy', 'sar', 'wire-fraud')][:12]

print(f'Selected {len(MY_BENCHMARK)} queries')
from collections import Counter
print('Doc types:', Counter(q['doc_type'] for q in MY_BENCHMARK))

In [ ]:
# ── Run benchmark on both indices ─────────────────────────────────────────────
dense = retriever_dense(field='body')

for idx in ('cortex-corpus', 'cortex-corpus-candidate'):
    results = run_queries_with_template(es, idx, MY_BENCHMARK, dense, k=5, n_passes=3)
    n5  = ndcg_at_k(results, MY_BENCHMARK, k=5)
    p95v = p95(results)
    print(f'{idx}: nDCG@5={n5:.3f}  p95={p95v:.0f}ms')

In [ ]:
# ── Save benchmark set ────────────────────────────────────────────────────────
pathlib.Path('/home/elastic/benchmark-set.json').write_text(json.dumps(MY_BENCHMARK, indent=2))
print(f'Saved {len(MY_BENCHMARK)} queries. Select Check.')